# Import Environment variables

In [1]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.ipynb

Found bucket: id=rw-migration-aou-rw-f7a4d148, bucketName=rw-migration-aou-rw-f7a4d148
-> Assigned migration variables (ID: rw-migration-aou-rw-f7a4d148)
Found bucket: id=temporary-workspace-bucket, bucketName=temporary-workspace-bucket-wb-perky-cabbage-8342
Found bucket: id=workspace-bucket, bucketName=workspace-bucket-wb-perky-cabbage-8342
✅ Successfully identified latest dataset: wb-silky-artichoke-2408.C2024Q3R9

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-perky-cabbage-8342
WORKSPACE_BUCKET: gs://workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_TEMP_BUCKET: gs://temporary-workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9
bucket_aou_tutorial: NOT FOUND
bucket_id_aou_tutorial: NOT FOUND
bucket_migrated: gs://rw-migration-aou-rw-f7a4d148
bucket_id_migrated: rw-migration-aou-rw-f7a4d148

✅ Saved to /home/jupyter/.bashrc
C2024Q3R9 BQ_DATASET
Multi-trait-GWAS-in-admixed-populations GIT_REPO
dataset_test2 BQ_DATASET
prep_C2024Q3R9 BQ_DATASET
rw-mig

In [2]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.ipynb

WORKSPACE_CDR = wb-silky-artichoke-2408.C2024Q3R9
WORKSPACE_BUCKET = gs://workspace-bucket-wb-perky-cabbage-8342
GOOGLE_PROJECT = wb-perky-cabbage-8342
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.R
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.sas


# Selection des SNPs du PRS313 version b38

## Download srNGS pvar for each chromosome from 1 to 22

In [ ]:
%%bash

BASE_URI="gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/pgen"
for chr in {1..22}; do
    echo "Downloading chr${chr}..."
    gsutil -u $GOOGLE_PROJECT cp \
        ${BASE_URI}/acaf_threshold.chr${chr}.pvar \
        srNGS/
done

## Copy prs313 list to bucket

In [4]:
import os
import subprocess

name_of_file_in_bucket = "prs313_b38_all_v2.csv"

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp 'Datas/{name_of_file_in_bucket}' '{my_bucket}/Data'")

print(f'[INFO] {name_of_file_in_bucket} is successfully uploaded into your working space')

Copying file://Datas/prs313_b38_all_v2.csv [Content-Type=text/csv]...
/ [1 files][  7.3 KiB/  7.3 KiB]                                                
Operation completed over 1 objects/7.3 KiB.                                      


[INFO] prs313_b38_all_v2.csv is successfully uploaded into your working space


## Try to find markers 

In [5]:
from pathlib import Path
import pandas as pd

# 1. Charger et nettoyer le fichier d'intervalles BED (prs313_b38_all.csv)
file_path = "Datas/prs313_b38_all_v2.csv"

# Lecture du fichier séparé par des points-virgules (;)
snps = pd.read_csv(file_path, sep="\t", header=None, dtype=str)

# Si une colonne d'index/numérotation est présente au début, on garde les 3 dernières colonnes
if snps.shape[1] > 3:
    snps = snps.iloc[:, -3:]

snps.columns = ["chr", "start", "end"]

# Nettoyage des chaînes : suppression des espaces et extraction du numéro de chromosome
snps["chr_clean"] = snps["chr"].str.replace("chr", "", case=False).str.strip()
snps["pos_clean"] = snps["end"].str.strip()

# Création du set des paires (chr, pos) cibles
target_pairs = set(zip(snps["chr_clean"], snps["pos_clean"]))

print(f"[INFO] Nombre de positions cibles uniques : {len(target_pairs)}")

# 2. Scanner les fichiers .pvar
matches = []

for n in range(1, 23):
    pvar_path = Path(f"srNGS/acaf_threshold.chr{n}.pvar")
    if not pvar_path.exists():
        print(f"[WARN] Fichier manquant : {pvar_path}")
        continue

    with pvar_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("#"):
                continue

            parts = line.strip().split()
            if len(parts) >= 5:
                chrom = parts[0].replace("chr", "").strip()
                pos = parts[1].strip()

                if (chrom, pos) in target_pairs:
                    matches.append(
                        {
                            "chr_pvar": chrom,
                            "pos_pvar": pos,
                            "id_pvar": parts[2],
                            "ref_pvar": parts[3],
                            "alt_pvar": parts[4],
                            "full_pvar_line": line.strip(),
                        }
                    )

df_matches = pd.DataFrame(matches)
print(f"[SUCCESS] Total de variants pvar retrouvés : {len(df_matches)}")

# 3. Fusion avec le DataFrame initial pour repérer les variants manquants
df_merged = snps.merge(
    df_matches,
    left_on=["chr_clean", "pos_clean"],
    right_on=["chr_pvar", "pos_pvar"],
    how="left",
)

# Afficher les variants absents
missing = df_merged[df_merged["id_pvar"].isna()]
print(
    f"\n[INFO] {len(missing)} variants n'ont pas été trouvés dans acaf_threshold :"
)
print(missing[["chr", "start", "end"]])

# Export du résultat final
df_merged.to_csv("Datas/snps_vs_pvar_python_join.csv", index=False)

[INFO] Nombre de positions cibles uniques : 311
[SUCCESS] Total de variants pvar retrouvés : 311

[INFO] 0 variants n'ont pas été trouvés dans acaf_threshold :
Empty DataFrame
Columns: [chr, start, end]
Index: []


In [6]:
!grep 145829794  srNGS/acaf_threshold.chr1.pvar

chr1	145829794	.	G	A,T	.	AC=246302,1;AF=0.297,1.205e-06;AN=829608;AS_QUALapprox=0|21738947|79;CALIBRATION_SENSITIVITY=0.4914,0.9961;QUALapprox=85;SCORE=-0.3497,-0.6765


## Data export

# Extraction des 313 variants sur les données srNGS

In [ ]:
%%bash

BASE_URI="gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/pgen"
DATAS_FOLDER="/home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Datas"
SRNGS_FILE="/home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS"
SRNGS_PRS313_FILE="/home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS_prs313"
mkdir -p srNGS_prs313

for chr in {1..22}; do
    echo "=== Traitement du Chromosome ${chr} ==="
    # 1. Téléchargement des 2 fichiers PLINK2 restants pour le chromosome
    gsutil -u $GOOGLE_PROJECT -m -o GSUtil:show_progress_bar=True cp ${BASE_URI}/acaf_threshold.chr${chr}.pgen ${SRNGS_FILE}/
    gsutil -u $GOOGLE_PROJECT -m -o GSUtil:show_progress_bar=True cp ${BASE_URI}/acaf_threshold.chr${chr}.psam ${SRNGS_FILE}/
    
    # 2. Extraction ciblée des 313 variants avec PLINK2
    plink2 --pfile ${SRNGS_FILE}/acaf_threshold.chr${chr} --extract bed1 ${DATAS_FOLDER}/prs313_b38_all_v2.csv --make-pgen --out ${SRNGS_PRS313_FILE}/prs313_all_chr${chr}
    
    # 3. Nettoyage immédiat du gros fichier .pgen brut pour économiser l'espace disque
    rm ${SRNGS_FILE}/acaf_threshold.chr${chr}.pgen
done

echo "[OK] Extraction terminée pour tous les chromosomes !"

## Fusion des fichiers en un seul `.bcf`

In [4]:
%%bash

SRNGS_PRS313_FILE="/home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS_prs313"

ls -v ${SRNGS_PRS313_FILE}/prs313_all_chr*.pgen | sed 's/.pgen//g' > ${SRNGS_PRS313_FILE}/list_concat_chrs.txt
plink2 --pmerge-list ${SRNGS_PRS313_FILE}/list_concat_chrs.txt --export bcf id-paste=iid --out ${SRNGS_PRS313_FILE}/prs313_all
bcftools index ${SRNGS_PRS313_FILE}/prs313_all.bcf

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS_prs313/prs313_all.log.
Options in effect:
  --export bcf id-paste=iid
  --out /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS_prs313/prs313_all
  --pmerge-list /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS_prs313/list_concat_chrs.txt

Start time: Sat Aug 29 20:16:33 2026
26036 MiB RAM detected, ~24877 available; reserving 13018 MiB for main
workspace.
Using up to 4 compute threads.
--pmerge-list: 22 filesets specified.
--pmerge-list: 414830 samples present.
--pmerge-list: Merged .psam written to
/home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS_prs313/prs313_all.psam
.
--pmerge-list: 22 .pvar files scanned, headers merged.
Concatenation job detected.
Concatenatin

In [6]:
%%bash

bcftools query -f '%CHROM\t%POS\t%ID\t%REF\t%ALT\n' srNGS_prs313/prs313_all.bcf | wc -l

311


In [7]:
import os
import subprocess

destination_filename = 'prs313_all.bcf'

# Récupère le nom du bucket Google Cloud depuis la variable d'environnement
my_bucket = os.getenv('WORKSPACE_BUCKET')

args = ["gsutil", "cp", f"srNGS_prs313/{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuelles erreurs retournées par gsutil
output.stderr

b'Copying file://srNGS_prs313/prs313_all.bcf [Content-Type=application/octet-stream]...\n/ [0 files][    0.0 B/ 32.8 MiB]                                                \r/ [1 files][ 32.8 MiB/ 32.8 MiB]                                                \r-\r\nOperation completed over 1 objects/32.8 MiB.                                     \n'